In [9]:
import pandas as pd

# ---------------------------------------------------
# Dataset
# ---------------------------------------------------
data = pd.DataFrame({
    "Outlook": ["Sunny", "Sunny", "Overcast", "Rain", "Rain",
                "Rain", "Overcast", "Sunny", "Sunny", "Rain",
                "Sunny", "Overcast", "Overcast", "Rain"],

    "Temperature": ["Hot", "Hot", "Hot", "Mild", "Cool",
                    "Cool", "Cool", "Mild", "Cool", "Mild",
                    "Mild", "Mild", "Hot", "Mild"],

    "Humidity": ["High", "High", "High", "High", "Normal",
                 "Normal", "Normal", "High", "Normal", "Normal",
                 "Normal", "High", "Normal", "High"],

    "Windy": [False, True, False, False, False,
              True, True, False, False, False,
              True, True, False, True],

    "Play": ["No", "No", "Yes", "Yes", "Yes",
             "No", "Yes", "No", "Yes", "Yes",
             "Yes", "Yes", "Yes", "No"]
})

print("Dataset:")
print(data)


# ---------------------------------------------------
# Entropy / Information Gain
# ---------------------------------------------------
import math

def entropy(values):
    total = len(values)

    if total == 0:
        return 0

    counts = values.value_counts()

    result = 0

    for count in counts:
        p = count / total
        result -= p * math.log2(p)

    return result


def information_gain(data, attribute, target="Play"):

    parent_entropy = entropy(data[target])

    weighted_entropy = 0

    for value in data[attribute].unique():

        subset = data[data[attribute] == value]

        weight = len(subset) / len(data)

        weighted_entropy += weight * entropy(subset[target])

    return parent_entropy - weighted_entropy


# ---------------------------------------------------
# Find best condition
# ---------------------------------------------------
def find_best_condition(data):

    attributes = [
        "Outlook",
        "Temperature",
        "Humidity",
        "Windy"
    ]

    best_attribute = None
    best_value = None
    best_gain = -1

    for attribute in attributes:

        for value in data[attribute].unique():

            subset = data[data[attribute] == value]

            if len(subset) == 0:
                continue

            gain = information_gain(data, attribute)

            if gain > best_gain:

                best_gain = gain
                best_attribute = attribute
                best_value = value

    return best_attribute, best_value


# ---------------------------------------------------
# Simple Rule Learner
# ---------------------------------------------------
def learn_rules(data):

    rules = []

    remaining = data.copy()

    while len(remaining) > 0:

        # If all examples are positive
        if all(remaining["Play"] == "Yes"):

            rules.append("IF conditions THEN Play = Yes")
            break

        attribute, value = find_best_condition(remaining)

        if attribute is None:
            break

        subset = remaining[
            remaining[attribute] == value
        ]

        positive = subset[subset["Play"] == "Yes"]

        # Create rule only if positive examples exist
        if len(positive) > 0:

            rule = (
                f"IF {attribute} = {value} "
                f"THEN Play = Yes"
            )

            rules.append(rule)

        # Remove covered examples
        remaining = remaining[
            remaining[attribute] != value
        ]

        # Stop if no examples remain
        if len(remaining) == 0:
            break

    return rules


# ---------------------------------------------------
# Generate Rules
# ---------------------------------------------------
rules = learn_rules(data)

print("\nLearned Classification Rules:")

for rule in rules:
    print(rule)



# ---------------------------------------------------
# Test Rules


# ---------------------------------------------------




Dataset:
     Outlook Temperature Humidity  Windy Play
0      Sunny         Hot     High  False   No
1      Sunny         Hot     High   True   No
2   Overcast         Hot     High  False  Yes
3       Rain        Mild     High  False  Yes
4       Rain        Cool   Normal  False  Yes
5       Rain        Cool   Normal   True   No
6   Overcast        Cool   Normal   True  Yes
7      Sunny        Mild     High  False   No
8      Sunny        Cool   Normal  False  Yes
9       Rain        Mild   Normal  False  Yes
10     Sunny        Mild   Normal   True  Yes
11  Overcast        Mild     High   True  Yes
12  Overcast         Hot   Normal  False  Yes
13      Rain        Mild     High   True   No

Learned Classification Rules:
IF Outlook = Sunny THEN Play = Yes
IF Windy = False THEN Play = Yes
IF conditions THEN Play = Yes
